## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [5]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [6]:
MY_BUBBLE_FILE = "anti_suveranist.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: anti_suveranist
Texte: 50


,id,agent,text
0,yt_Tx8GhU2LeyI_UgwoWOyzF2UbPYnguUB4AaABAg,Anti-suveranist,Am toată încrederea că oameni ( de bine ) ca :...
1,yt_6_Hc2S02Duw_Ugytw6-BDQ2pA_Zi-TB4AaABAg,Anti-suveranist,Apropo de avalansa de troli ce se devarsa si a...
2,yt_bee6nXyzJ_E_UgxQ1N1kdP_MTx8B4K14AaABAg,Anti-suveranist,Aceasta nu este o emisiune....este o regizare ...
3,yt_bee6nXyzJ_E_Ugyjnx0utsCXEXc94q54AaABAg,Anti-suveranist,La pregatit bine Putin a investit bani in Guru...
4,yt_im3QoqSgfDo_UgwOL2VI3_toiLZSDqh4AaABAg,Anti-suveranist,"Eu am vorbit cu susținători de ai lui CG, îs d..."


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [7]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Am toată încrederea că oameni ( de bine ) ca : G Simion , Călinge , dna Găurilă ...... vor avea mare grijă să pună fie piedici , fie bețe-n roate astfel încât să rămanem sub tutela cremlinului


In [8]:
texts[0]

'Am toată încrederea că oameni ( de bine ) ca : G Simion , Călinge , dna Găurilă ...... vor avea mare grijă să pună fie piedici , fie bețe-n roate astfel încât să rămanem sub tutela cremlinului'

In [9]:
metadata[0]

{'id': 'yt_Tx8GhU2LeyI_UgwoWOyzF2UbPYnguUB4AaABAg',
 'text': 'Am toată încrederea că oameni ( de bine ) ca : G Simion , Călinge , dna Găurilă ...... vor avea mare grijă să pună fie piedici , fie bețe-n roate astfel încât să rămanem sub tutela cremlinului',
 'source_channel': 'AlephNewsOfficial',
 'channel_family': 'mainstream',
 'video_title': 'ATENȚIE: România e „binevenită” să aplice iar pentru Visa Waiver, spune Ambasadorul SUA la București',
 'target_refined': 'simion',
 'stance_to_target': 'anti',
 'confidence': 0.9,
 'discourse_type': 'T3_opozitie_suveranista',
 'discourse_subtype': 'opozitie_difuza',
 'type_confidence': 'medium',
 'agent': 'Anti-suveranist',
 'slug': 'anti_suveranist',
 'personality': 'critic, vigilent, defensiv',
 'speaks': 'contestatar, mai argumentativ',
 'definition': 'respinge liderii și discursul suveranist'}

## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [10]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.80it/s]

Număr texte: 50
Dimensiune embeddings: (50, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [ ]:
# TODO student:
# Bula mea are 50 texte.
# Au fost generați 50 vectori.
# A doua valoare din embeddings.shape reprezintă (384) reprezinta lungimea vectorului

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [12]:
index = faiss.IndexFlatIP(embeddings.shape[1])

index.add(embeddings)

out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\anti_suveranist
Vectori în index: 50


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [ ]:
# TODO student:
# index.faiss există: da
# index.pkl există: da
# index.ntotal este egal cu numărul de texte: 50

In [13]:
print("Numar vectori in index:", index.ntotal)
print("Dimensiunea fiecarui vector:", index.d)
print("Tip in index:", type(index))

Numar vectori in index: 50
Dimensiunea fiecarui vector: 384
Tip in index: <class 'faiss.swigfaiss_avx2.IndexFlatIP'>


## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În următorul continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [17]:
# Text nou introdus în aplicație

input_text = "Cine poate vota un manipulator care se vrea a fi liderul poporului ca Simion?"

In [18]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [11]:
# query_vector

In [19]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.673
Text: Eu NU VOTEZ SIMION, pentru că iau în considerare spusele lui Călin Georgescu despre Simion în NOIEMBRIE 2024 : ,,Domnilor și doamnelor din partidul AUR. Vă întreb ,sincer, sunteți siguri ca George Simion este omul care poate conduce România? Stiți de ce nu am acceptat propunerile lui George Simion de a fi premier ,sau de a candida din partea AUR la ,,preșidenție,,(aici habar n-are că este vorba de ,,președinție)? Pentru ca George Simion nu este omul oamenilor lui, al celor care cu adevărat îl su

Rezultat 2
Scor: 0.612
Text: Cred ca AUR-ul e trans in jos chiar de Simion,cand vorbim de imaginea partidului si increderea pe care o emana,lumea am vazut ca se uita la varf si identifica partidul cu seful acestuia...iar in cazul asta Simion nu prea mai ajuta...zic si eu.

Rezultat 3
Scor: 0.61
Text: Pare adus cu forta, fara sotie, fara un sunet. In toata compania electorala, un cuvant nu a spus despre tine. O singura data nu a pronuntat " Votati George Simion!" N

### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante;
- dacă textele recuperate exprimă vocea agentului;
- dacă ai observat un text slab care ar trebui eliminat.

Toate rezultatele sunt destul de relevante, cu mici retineri pentru cazul 5 unde textul nu contine argumente directe, ci doar face apel la un joc idealitic, raspuns ce ar putea fi potential eliminat. Vocea agentului este in mare parte regasita in aceste raspunsuri.